In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from pathlib import Path
import pickle
from time import time

In [17]:
processed_data_dir = Path("../data/processed")
models_dir = Path("../models")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

In [4]:
X_train_tfidf = load_npz(processed_data_dir / "X_train_tfidf.npz")
X_val_tfidf = load_npz(processed_data_dir / "X_val_tfidf.npz")
X_test_tfidf = load_npz(processed_data_dir / "X_test_tfidf.npz")

In [5]:
X_train_seq = np.load(processed_data_dir / "X_train_seq.npy")
X_val_seq = np.load(processed_data_dir / "X_val_seq.npy")
X_test_seq = np.load(processed_data_dir / "X_test_seq.npy")

In [6]:
y_train = pd.read_csv(processed_data_dir / "y_train.csv")["label"].values
y_val = pd.read_csv(processed_data_dir / "y_val.csv")["label"].values
y_test = pd.read_csv(processed_data_dir / "y_test.csv")["label"].values

In [7]:
X_train_tfidf.shape, X_train_seq.shape, y_train.shape

((65988, 30000), (65988, 500), (65988,))

#### Logistic Regression

In [ ]:
params = {
    "solver": ["lbfgs", "liblinear"],
    "C": [0.01, 0.1, 1, 10]
}

lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_grid = GridSearchCV(lr_base, params, cv=5, n_jobs=-1, verbose=1)

lr_tarining_start = time()
lr_grid.fit(X_train_tfidf, y_train)
lr_training_end = time()
lr_training_time = lr_training_end - lr_tarining_start

lr_grid.score(X_val_tfidf, y_val)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
0.988968359801188



In [ ]:
lr = lr_grid.best_estimator_
lr_ytrain_pred = lr.predict(X_train_tfidf)
lr_yval_pred = lr.predict(X_val_tfidf)
lr_ytest_pred = lr.predict(X_test_tfidf)

In [ ]:
pickle.dump(lr, open(models_dir / "lr.pkl", "wb"))

pd.Series(lr_ytrain_pred).to_csv(processed_data_dir / "lr_ytrain_pred.csv", index=False)
pd.Series(lr_yval_pred).to_csv(processed_data_dir / "lr_yval_pred.csv", index=False)
pd.Series(lr_ytest_pred).to_csv(processed_data_dir / "lr_ytest_pred.csv", index=False)

#### FNN

In [13]:
fnn = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_seq.shape[1],)),
    tf.keras.layers.Embedding(input_dim=30000, output_dim=64),
    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(1, activation="linear")
])

In [14]:
fnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 64)        │     1,920,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,923,169 (7.34 MB)

 Trainable params: 1,923,169 (7.34 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
fnn.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
fnn_training_start = time()
history = fnn.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
fnn_training_end = time()
fnn_training_time = fnn_training_end - fnn_training_start


Epoch 1/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.8551 - loss: 0.2750 - val_accuracy: 0.9613 - val_loss: 0.1122
Epoch 2/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9584 - loss: 0.1152 - val_accuracy: 0.9806 - val_loss: 0.0624
Epoch 3/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9681 - loss: 0.0880 - val_accuracy: 0.9171 - val_loss: 0.1557
Epoch 4/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9738 - loss: 0.0734 - val_accuracy: 0.9857 - val_loss: 0.0523
Epoch 5/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9760 - loss: 0.0650 - val_accuracy: 0.9731 - val_loss: 0.0682
Epoch 6/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9802 - loss: 0.0559 - val_accuracy: 0.9791 - val_loss: 0.0687
Epoch 7/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9803 - loss: 0.0558 - val_accuracy: 0.9315 - val_loss: 0.1563



In [ ]:
history_fnn = history.history
plt.plot(history_fnn["loss"], label="train_loss")
plt.plot(history_fnn["val_loss"], label="val_loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("FNN Loss Curves")
plt.savefig(results_dir / "fnn_loss_curves.png")

In [ ]:
fnn.save(models_dir / "fnn.keras")

fnn_ytrain_pred = (fnn.predict(X_train_seq) > 0.5).astype(int).flatten()
fnn_yval_pred = (fnn.predict(X_val_seq) > 0.5).astype(int).flatten()
fnn_ytest_pred = (fnn.predict(X_test_seq) > 0.5).astype(int).flatten()
pd.Series(fnn_ytrain_pred).to_csv(processed_data_dir / "fnn_ytrain_pred.csv", index=False)
pd.Series(fnn_yval_pred).to_csv(processed_data_dir / "fnn_yval_pred.csv", index=False)
pd.Series(fnn_ytest_pred).to_csv(processed_data_dir / "fnn_ytest_pred.csv", index=False)


2063/2063 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step



#### RNN

In [20]:
rnn = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_seq.shape[1],)),
    tf.keras.layers.Embedding(input_dim=30000, output_dim=64),

    tf.keras.layers.SimpleRNN(32, activation="sigmoid", return_sequences=False),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(1, activation="linear")
])

In [21]:
rnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 64)        │     1,920,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,923,137 (7.34 MB)

 Trainable params: 1,923,137 (7.34 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
rnn.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
start_training_rnn = time()
history = rnn.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
rnn_training_end = time()
rnn_training_time = rnn_training_end - start_training_rnn


Epoch 1/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 74s 35ms/step - accuracy: 0.4813 - loss: 0.6921 - val_accuracy: 0.4835 - val_loss: 0.6836
Epoch 2/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 69s 34ms/step - accuracy: 0.4799 - loss: 0.6804 - val_accuracy: 0.4836 - val_loss: 0.6768
Epoch 3/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 82s 34ms/step - accuracy: 0.4887 - loss: 0.6696 - val_accuracy: 0.4900 - val_loss: 0.6740
Epoch 4/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 69s 33ms/step - accuracy: 0.4925 - loss: 0.6638 - val_accuracy: 0.4889 - val_loss: 0.6757
Epoch 5/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 69s 34ms/step - accuracy: 0.4926 - loss: 0.6626 - val_accuracy: 0.4877 - val_loss: 0.6815
Epoch 6/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 82s 34ms/step - accuracy: 0.4932 - loss: 0.6608 - val_accuracy: 0.4899 - val_loss: 0.6810



In [ ]:
history_rnn = history.history
plt.plot(history_rnn["loss"], label="train_loss")
plt.plot(history_rnn["val_loss"], label="val_loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("RNN Loss Curves")
plt.savefig(results_dir / "rnn_loss_curves.png")

In [ ]:
rnn.save(models_dir / "rnn.keras")

rnn_ytrain_pred = (rnn.predict(X_train_seq) > 0.5).astype(int).flatten()
rnn_yval_pred = (rnn.predict(X_val_seq) > 0.5).astype(int).flatten()
rnn_ytest_pred = (rnn.predict(X_test_seq) > 0.5).astype(int).flatten()
pd.Series(rnn_ytrain_pred).to_csv(processed_data_dir / "rnn_ytrain_pred.csv", index=False)
pd.Series(rnn_yval_pred).to_csv(processed_data_dir / "rnn_yval_pred.csv", index=False)
pd.Series(rnn_ytest_pred).to_csv(processed_data_dir / "rnn_ytest_pred.csv", index=False)


2063/2063 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step



#### LSTM

In [26]:
lstm = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_seq.shape[1],)),
    tf.keras.layers.Embedding(input_dim=30000, output_dim=64),

    tf.keras.layers.LSTM(32, activation="sigmoid", return_sequences=False),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(1, activation="linear")
])

In [27]:
lstm.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 64)        │     1,920,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,932,449 (7.37 MB)

 Trainable params: 1,932,449 (7.37 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
start_training_lstm = time()
history = lstm.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
lstm_training_end = time()
lstm_training_time = lstm_training_end - start_training_lstm


Epoch 1/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 90s 42ms/step - accuracy: 0.4805 - loss: 0.6887 - val_accuracy: 0.4835 - val_loss: 0.6794
Epoch 2/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 85s 41ms/step - accuracy: 0.4827 - loss: 0.6749 - val_accuracy: 0.4910 - val_loss: 0.6698
Epoch 3/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 84s 41ms/step - accuracy: 0.5749 - loss: 0.6138 - val_accuracy: 0.9301 - val_loss: 0.2343
Epoch 4/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 84s 41ms/step - accuracy: 0.9662 - loss: 0.1419 - val_accuracy: 0.9777 - val_loss: 0.0896
Epoch 5/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 84s 41ms/step - accuracy: 0.9828 - loss: 0.0709 - val_accuracy: 0.9862 - val_loss: 0.0431
Epoch 6/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 85s 41ms/step - accuracy: 0.9944 - loss: 0.0221 - val_accuracy: 0.9891 - val_loss: 0.0400
Epoch 7/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 86s 41ms/step - accuracy: 0.9969 - loss: 0.0116 - val_accuracy: 0.9882 - val_loss: 0.0468
Epoch 8/10
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 85s 41ms/step - accuracy: 0.9983 

In [ ]:
history_lstm = history.history
plt.plot(history_lstm["loss"], label="train_loss")
plt.plot(history_lstm["val_loss"], label="val_loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("LSTM Loss Curves")
plt.savefig(results_dir / "lstm_loss_curves.png")

In [ ]:
lstm.save(models_dir / "lstm.keras")

lstm_ytrain_pred = (lstm.predict(X_train_seq) > 0.5).astype(int).flatten()
lstm_yval_pred = (lstm.predict(X_val_seq) > 0.5).astype(int).flatten()
lstm_ytest_pred = (lstm.predict(X_test_seq) > 0.5).astype(int).flatten()
pd.Series(lstm_ytrain_pred).to_csv(processed_data_dir / "lstm_ytrain_pred.csv", index=False)
pd.Series(lstm_yval_pred).to_csv(processed_data_dir / "lstm_yval_pred.csv", index=False)
pd.Series(lstm_ytest_pred).to_csv(processed_data_dir / "lstm_ytest_pred.csv", index=False)


2063/2063 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step

